In [1]:
from sklearn.model_selection import train_test_split
import optuna
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.impute import SimpleImputer
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor

In [2]:
df = pd.read_csv("best_data.csv")

In [3]:
hold = df['Percent_Bleaching'] <= 1
df_train, df_test, _, _ = train_test_split(df, hold, test_size=0.2, random_state=42, stratify= hold)

In [8]:
'''
df_1_0 = df[df.Percent_Bleaching <= 1].copy()
df_1 = df[df.Percent_Bleaching > 1].copy()
df_1_0 = df_1_0.sample(n = 2000, random_state= 42) # Adding 2000 for training reg if there classif is error
'''
df_classif = df_train.copy()
df_reg = df_train.copy()

# Classification

In [9]:
df_classif['Status'] = df_classif['Percent_Bleaching'] <= 1
X = df_classif.drop(columns=["Percent_Bleaching", "Status"])
Y = df_classif["Status"]

In [12]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

## PCA

In [13]:
X_train_raw_month, X_test_raw_month = X_train_raw[['Month_1', 'Month_2', 'Month_3', 'Month_4', 'Month_5', 'Month_6',
       'Month_7', 'Month_8', 'Month_9', 'Month_10', 'Month_11', 'Month_12']].copy(), X_test_raw[['Month_1', 'Month_2', 'Month_3', 'Month_4', 'Month_5', 'Month_6',
       'Month_7', 'Month_8', 'Month_9', 'Month_10', 'Month_11', 'Month_12']].copy()

X_train_raw_fe, X_test_fe = X_train_raw[['Latitude_Degrees', 'Longitude_Degrees', 'Depth_m', 'ClimSST', 'SSTA']].copy(), \
       X_test_raw[['Latitude_Degrees', 'Longitude_Degrees', 'Depth_m', 'ClimSST', 'SSTA']].copy()


X_train_raw = X_train_raw.drop(columns= ['Month_1', 'Month_2', 'Month_3', 'Month_4', 'Month_5', 'Month_6',
       'Month_7', 'Month_8', 'Month_9', 'Month_10', 'Month_11', 'Month_12'])
X_test_raw = X_test_raw.drop(columns= ['Month_1', 'Month_2', 'Month_3', 'Month_4', 'Month_5', 'Month_6',
       'Month_7', 'Month_8', 'Month_9', 'Month_10', 'Month_11', 'Month_12'])

imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train_raw)
X_test_imp = imputer.transform(X_test_raw)

# SEPARATE imputer for FE — and keep as DataFrame
fe_imputer = SimpleImputer(strategy='median')
X_train_raw_fe = pd.DataFrame(
    fe_imputer.fit_transform(X_train_raw_fe),
    columns=X_train_raw_fe.columns,
    index=X_train_raw_fe.index
)
X_test_fe = pd.DataFrame(
    fe_imputer.transform(X_test_fe),
    columns=X_test_fe.columns,
    index=X_test_fe.index
)

scaler = StandardScaler()
X_train_ss = scaler.fit_transform(X_train_imp)
X_test_ss = scaler.transform(X_test_imp)

pca = PCA(n_components=0.75)
X_train_pca = pca.fit_transform(X_train_ss)
X_test_pca = pca.transform(X_test_ss)

X_train_pca_df = pd.DataFrame(X_train_pca, columns=[f'PC{i+1}' for i in range(X_train_pca.shape[1])])
X_test_pca_df = pd.DataFrame(X_test_pca, columns=[f'PC{i+1}' for i in range(X_test_pca.shape[1])])

X_train_raw_month = X_train_raw_month.reset_index(drop=True)
X_test_raw_month = X_test_raw_month.reset_index(drop=True)

X_train_raw_fe = X_train_raw_fe.reset_index(drop=True)
X_test_fe = X_test_fe.reset_index(drop=True)

X_train_pca = pd.concat([X_train_pca_df, X_train_raw_month, X_train_raw_fe], axis=1)
X_test_pca  = pd.concat([X_test_pca_df,  X_test_raw_month,  X_test_fe],       axis=1)

## Optuna

In [14]:
def cv_classif(model_factory, X_num, X_extras, y, cv_splits=5):
    X_num    = X_num.reset_index(drop=True)
    X_extras = X_extras.reset_index(drop=True)
    y        = y.reset_index(drop=True)
    
    skf = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=42)
    scores = []
    
    for train_idx, val_idx in skf.split(X_num, y):
        # ----- split -----
        X_num_tr, X_num_val = X_num.iloc[train_idx], X_num.iloc[val_idx]
        X_ex_tr  = X_extras.iloc[train_idx].reset_index(drop=True)
        X_ex_val = X_extras.iloc[val_idx].reset_index(drop=True)
        y_tr     = y.iloc[train_idx].reset_index(drop=True)
        y_val    = y.iloc[val_idx].reset_index(drop=True)
        
        # ----- fit preprocessing บน train fold เท่านั้น ⭐ -----
        imputer = SimpleImputer(strategy='median')
        X_tr_imp  = imputer.fit_transform(X_num_tr)
        X_val_imp = imputer.transform(X_num_val)
        
        scaler = StandardScaler()
        X_tr_ss  = scaler.fit_transform(X_tr_imp)
        X_val_ss = scaler.transform(X_val_imp)
        
        pca = PCA(n_components=0.75)
        X_tr_pca  = pca.fit_transform(X_tr_ss)
        X_val_pca = pca.transform(X_val_ss)
        
        # ----- reattach extras -----
        n_pcs = X_tr_pca.shape[1]
        pc_cols = [f'PC{i+1}' for i in range(n_pcs)]
        X_tr_final  = pd.concat([pd.DataFrame(X_tr_pca,  columns=pc_cols), X_ex_tr],  axis=1)
        X_val_final = pd.concat([pd.DataFrame(X_val_pca, columns=pc_cols), X_ex_val], axis=1)
        
        # ----- train + evaluate -----
        model = model_factory()    # สร้าง model ใหม่ทุก fold
        model.fit(X_tr_final, y_tr)
        y_pred = model.predict(X_val_final)
        scores.append(f1_score(y_val, y_pred, average='weighted'))
    
    return np.mean(scores)


X_extras_train = pd.concat([
    X_train_raw_month.reset_index(drop=True),
    X_train_raw_fe.reset_index(drop=True)
], axis=1)


# ============================================================
# 1. Random Forest
# ============================================================
def objective_rf(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 200, 600),
        'max_depth':         trial.suggest_int('max_depth', 10, 50),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 1, 5),
        'max_features':      trial.suggest_categorical('max_features', ['sqrt', None]),
        'random_state':      42,
        'n_jobs':            -1,
    }
    
    def make_model():
        return RandomForestClassifier(**params)
    
    return cv_classif(make_model, X_train_raw, X_extras_train, y_train)

studies = {}
n_trials = 50

# ---------- Random Forest ----------
print("=" * 60)
print("🌲 Tuning Random Forest...")
print("=" * 60)
studies['rf'] = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42)
)
studies['rf'].optimize(objective_rf, n_trials=n_trials, show_progress_bar=True)

# ============================================================
# Summary
# ============================================================
print("\n" + "=" * 60)
print("📊 Best Results")
print("=" * 60)
for name, study in studies.items():
    print(f"\n{name.upper()}:")
    print(f"  Best F1:     {study.best_value:.4f}")
    print(f"  Best params: {study.best_params}")

[I 2026-05-09 11:04:23,920] A new study created in memory with name: no-name-c03ef7be-d014-4d11-9703-02e8266ea94b


🌲 Tuning Random Forest...


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-05-09 11:04:28,676] Trial 0 finished with value: 0.8849134760245114 and parameters: {'n_estimators': 350, 'max_depth': 48, 'min_samples_split': 8, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.8849134760245114.
[I 2026-05-09 11:04:39,828] Trial 1 finished with value: 0.8834305593438028 and parameters: {'n_estimators': 223, 'max_depth': 45, 'min_samples_split': 7, 'min_samples_leaf': 4, 'max_features': None}. Best is trial 0 with value: 0.8849134760245114.
[I 2026-05-09 11:05:06,704] Trial 2 finished with value: 0.8999445450524999 and parameters: {'n_estimators': 533, 'max_depth': 18, 'min_samples_split': 3, 'min_samples_leaf': 1, 'max_features': None}. Best is trial 2 with value: 0.8999445450524999.
[I 2026-05-09 11:05:25,471] Trial 3 finished with value: 0.8991018045133814 and parameters: {'n_estimators': 373, 'max_depth': 21, 'min_samples_split': 7, 'min_samples_leaf': 1, 'max_features': None}. Best is trial 2 with value: 0.8999445450524999.
[I

In [ ]:
# {'n_estimators': 512, 'max_depth': 47, 'min_samples_split': 3, 'min_samples_leaf': 1, 'max_features': 'sqrt'}
rf_clf_model = RandomForestClassifier(
    **studies["rf"].best_params,
    n_jobs=-1, 
    random_state=42
)

In [ ]:
rf_clf_model.fit(X_train_pca, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",512
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",47
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",3
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

# Regression

In [15]:
X = df_reg.drop("Percent_Bleaching", axis=1)
Y = df_reg["Percent_Bleaching"]

In [16]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

## PCA

In [17]:
X_train_raw_month, X_test_raw_month = X_train_raw[['Month_1', 'Month_2', 'Month_3', 'Month_4', 'Month_5', 'Month_6',
       'Month_7', 'Month_8', 'Month_9', 'Month_10', 'Month_11', 'Month_12']].copy(), X_test_raw[['Month_1', 'Month_2', 'Month_3', 'Month_4', 'Month_5', 'Month_6',
       'Month_7', 'Month_8', 'Month_9', 'Month_10', 'Month_11', 'Month_12']].copy()

X_train_raw_fe, X_test_fe = X_train_raw[['Latitude_Degrees', 'Longitude_Degrees', 'Depth_m', 'ClimSST', 'SSTA']].copy(), \
       X_test_raw[['Latitude_Degrees', 'Longitude_Degrees', 'Depth_m', 'ClimSST', 'SSTA']].copy()


X_train_raw = X_train_raw.drop(columns= ['Month_1', 'Month_2', 'Month_3', 'Month_4', 'Month_5', 'Month_6',
       'Month_7', 'Month_8', 'Month_9', 'Month_10', 'Month_11', 'Month_12'])
X_test_raw = X_test_raw.drop(columns= ['Month_1', 'Month_2', 'Month_3', 'Month_4', 'Month_5', 'Month_6',
       'Month_7', 'Month_8', 'Month_9', 'Month_10', 'Month_11', 'Month_12'])

imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train_raw)
X_test_imp = imputer.transform(X_test_raw)

# SEPARATE imputer for FE — and keep as DataFrame
fe_imputer = SimpleImputer(strategy='median')
X_train_raw_fe = pd.DataFrame(
    fe_imputer.fit_transform(X_train_raw_fe),
    columns=X_train_raw_fe.columns,
    index=X_train_raw_fe.index
)
X_test_fe = pd.DataFrame(
    fe_imputer.transform(X_test_fe),
    columns=X_test_fe.columns,
    index=X_test_fe.index
)

scaler = StandardScaler()
X_train_ss = scaler.fit_transform(X_train_imp)
X_test_ss = scaler.transform(X_test_imp)

pca = PCA(n_components=0.75)
X_train_pca = pca.fit_transform(X_train_ss)
X_test_pca = pca.transform(X_test_ss)

X_train_pca_df = pd.DataFrame(X_train_pca, columns=[f'PC{i+1}' for i in range(X_train_pca.shape[1])])
X_test_pca_df = pd.DataFrame(X_test_pca, columns=[f'PC{i+1}' for i in range(X_test_pca.shape[1])])

X_train_raw_month = X_train_raw_month.reset_index(drop=True)
X_test_raw_month = X_test_raw_month.reset_index(drop=True)

X_train_raw_fe = X_train_raw_fe.reset_index(drop=True)
X_test_fe = X_test_fe.reset_index(drop=True)

X_train_pca = pd.concat([X_train_pca_df, X_train_raw_month, X_train_raw_fe], axis=1)
X_test_pca  = pd.concat([X_test_pca_df,  X_test_raw_month,  X_test_fe],       axis=1)

## Optuna

In [18]:
# ── prepare a version of X_train that already has month columns ──
month_cols = ['Month_1','Month_2','Month_3','Month_4','Month_5','Month_6',
              'Month_7','Month_8','Month_9','Month_10','Month_11','Month_12',
              'Latitude_Degrees', 'Longitude_Degrees', 'Depth_m', 'ClimSST', 'SSTA']

X_train_no_month = X_train_raw.copy() 
X_train_month = pd.concat([
    X_train_raw_month.reset_index(drop=True),
    X_train_raw_fe.reset_index(drop=True)
], axis=1)

# helper: run CV with month columns reattached after PCA inside each fold
def cv_with_month(pipeline, X_num, X_mon, y, cv=4):
    from sklearn.model_selection import KFold

    # Reset all indices upfront so iloc slicing stays aligned
    X_num = X_num.reset_index(drop=True)
    X_mon = X_mon.reset_index(drop=True)
    y     = y.reset_index(drop=True)

    kf = KFold(n_splits=cv, shuffle=True, random_state=42)
    scores = []
    for train_idx, val_idx in kf.split(X_num):
        X_num_tr, X_num_val = X_num.iloc[train_idx], X_num.iloc[val_idx]
        X_mon_tr  = X_mon.iloc[train_idx].reset_index(drop=True)
        X_mon_val = X_mon.iloc[val_idx].reset_index(drop=True)
        y_tr  = y.iloc[train_idx].reset_index(drop=True)
        y_val = y.iloc[val_idx].reset_index(drop=True)

        # fit the pipeline (impute+scale+pca) on numeric only
        steps_no_model = Pipeline(pipeline.steps[:-1])
        X_tr_pca  = steps_no_model.fit_transform(X_num_tr)
        X_val_pca = steps_no_model.transform(X_num_val)

        n_pcs = X_tr_pca.shape[1]
        pc_cols = [f'PC{i+1}' for i in range(n_pcs)]

        X_tr_final  = pd.concat([pd.DataFrame(X_tr_pca,  columns=pc_cols), X_mon_tr],  axis=1)
        X_val_final = pd.concat([pd.DataFrame(X_val_pca, columns=pc_cols), X_mon_val], axis=1)

        model = pipeline.steps[-1][1]
        model.fit(X_tr_final, y_tr)
        scores.append(model.score(X_val_final, y_val))   # R²
    return np.mean(scores)

In [19]:
def et_objective(trial):
    model = ExtraTreesRegressor(
        n_estimators     = trial.suggest_int("n_estimators", 50, 300),
        max_depth        = trial.suggest_int("max_depth", 5, 40),
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 5),
        max_features     = trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        n_jobs=-1, random_state=42
    )
    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('pca',     PCA(n_components=0.75)),
        ('model',   model)
    ])
    return cv_with_month(pipeline, X_train_no_month, X_train_month, y_train)

et_study = optuna.create_study(direction="maximize")
et_study.optimize(et_objective, n_trials=50)
print("========== ET BEST ==========")
print(et_study.best_params)
print(f"Best R²: {et_study.best_value:.4f}")

[I 2026-05-09 11:14:22,724] A new study created in memory with name: no-name-68001e00-13b3-44ec-854e-a9cad4a0f2de
[I 2026-05-09 11:14:23,351] Trial 0 finished with value: 0.25914521518406386 and parameters: {'n_estimators': 82, 'max_depth': 10, 'min_samples_leaf': 5, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.25914521518406386.
[I 2026-05-09 11:14:24,357] Trial 1 finished with value: 0.4972592969098622 and parameters: {'n_estimators': 171, 'max_depth': 16, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.4972592969098622.
[I 2026-05-09 11:14:25,099] Trial 2 finished with value: 0.4139378962160065 and parameters: {'n_estimators': 93, 'max_depth': 25, 'min_samples_leaf': 5, 'max_features': 'log2'}. Best is trial 1 with value: 0.4972592969098622.
[I 2026-05-09 11:14:26,257] Trial 3 finished with value: 0.5409578897193698 and parameters: {'n_estimators': 189, 'max_depth': 32, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 3 with value

========== ET BEST ==========
{'n_estimators': 243, 'max_depth': 28, 'min_samples_leaf': 1, 'max_features': 'log2'}
Best R²: 0.6104


In [20]:
def rf_objective(trial):
    model = RandomForestRegressor(
        n_estimators      = trial.suggest_int("n_estimators", 50, 300),
        max_depth         = trial.suggest_int("max_depth", 5, 40),
        min_samples_split = trial.suggest_int("min_samples_split", 2, 10),
        min_samples_leaf  = trial.suggest_int("min_samples_leaf", 1, 5),
        max_features      = trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        n_jobs=-1, random_state=42
    )
    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('pca',     PCA(n_components=0.75)),
        ('model',   model)
    ])
    return cv_with_month(pipeline, X_train_no_month, X_train_month, y_train)

rf_study = optuna.create_study(direction="maximize")
rf_study.optimize(rf_objective, n_trials=50)
print("========== RF BEST ==========")
print(rf_study.best_params)
print(f"Best R²: {rf_study.best_value:.4f}")

[I 2026-05-09 11:15:27,685] A new study created in memory with name: no-name-e9b30180-5dff-474d-9b41-57bf5fabfe2f
[I 2026-05-09 11:15:30,632] Trial 0 finished with value: 0.5534467855323508 and parameters: {'n_estimators': 287, 'max_depth': 25, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': 'log2'}. Best is trial 0 with value: 0.5534467855323508.
[I 2026-05-09 11:15:32,587] Trial 1 finished with value: 0.5663922955635365 and parameters: {'n_estimators': 179, 'max_depth': 21, 'min_samples_split': 8, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.5663922955635365.
[I 2026-05-09 11:15:34,796] Trial 2 finished with value: 0.5629781832742434 and parameters: {'n_estimators': 222, 'max_depth': 25, 'min_samples_split': 9, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 1 with value: 0.5663922955635365.
[I 2026-05-09 11:15:35,942] Trial 3 finished with value: 0.27268051734841414 and parameters: {'n_estimators': 154, 'max_depth': 5, 'm

========== RF BEST ==========
{'n_estimators': 285, 'max_depth': 28, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt'}
Best R²: 0.5946


In [21]:
def hgb_objective(trial):
    model = HistGradientBoostingRegressor(
        max_iter          = trial.suggest_int  ("max_iter",          50, 400),
        max_depth         = trial.suggest_int  ("max_depth",          3,  15),
        learning_rate     = trial.suggest_float("learning_rate",  0.005, 0.1),
        min_samples_leaf  = trial.suggest_int  ("min_samples_leaf",   1,  50),
        l2_regularization = trial.suggest_float("l2_regularization", 0.0, 1.0),
        random_state=42
    )
    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('pca',     PCA(n_components=0.75)),
        ('model',   model)
    ])
    return cv_with_month(pipeline, X_train_no_month, X_train_month, y_train)

hgb_study = optuna.create_study(direction="maximize")
hgb_study.optimize(hgb_objective, n_trials=50)
print("========== HGB BEST ==========")
print(hgb_study.best_params)
print(f"Best R²: {hgb_study.best_value:.4f}")

[I 2026-05-09 11:17:24,968] A new study created in memory with name: no-name-edef252d-5931-4040-b552-ff793915a984
[I 2026-05-09 11:17:26,407] Trial 0 finished with value: 0.5408069695863039 and parameters: {'max_iter': 155, 'max_depth': 9, 'learning_rate': 0.08172462904159584, 'min_samples_leaf': 17, 'l2_regularization': 0.06542121565738257}. Best is trial 0 with value: 0.5408069695863039.
[I 2026-05-09 11:17:28,295] Trial 1 finished with value: 0.5583487618730447 and parameters: {'max_iter': 395, 'max_depth': 13, 'learning_rate': 0.09807532784672798, 'min_samples_leaf': 26, 'l2_regularization': 0.43200009622308433}. Best is trial 1 with value: 0.5583487618730447.
[I 2026-05-09 11:17:30,159] Trial 2 finished with value: 0.5488151362998868 and parameters: {'max_iter': 322, 'max_depth': 9, 'learning_rate': 0.055030719246068505, 'min_samples_leaf': 25, 'l2_regularization': 0.9553209738575233}. Best is trial 1 with value: 0.5583487618730447.
[I 2026-05-09 11:17:32,110] Trial 3 finished wit

========== HGB BEST ==========
{'max_iter': 393, 'max_depth': 11, 'learning_rate': 0.07113754350959134, 'min_samples_leaf': 3, 'l2_regularization': 0.3400646756584348}
Best R²: 0.5667


In [ ]:
# {'n_estimators': 212, 'max_depth': 30, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2'}
rf_reg_model = RandomForestRegressor(
    **studies["rf"].best_params,
    n_jobs=-1, 
    random_state=42
)

# {'n_estimators': 280, 'max_depth': 30, 'min_samples_leaf': 1, 'max_features': 'sqrt'}
et_reg_model = ExtraTreesRegressor(
    **studies["rf"].best_params,
    n_jobs=-1, 
    random_state=42
)

# {'max_iter': 386, 'max_depth': 13, 'learning_rate': 0.09761375670573119, 'min_samples_leaf': 23, 'l2_regularization': 0.8265105894755381}
hgb_reg_model = HistGradientBoostingRegressor(
    **studies["rf"].best_params,
    n_jobs=-1, 
    random_state=42
)